#**Synthetic Outpatient Clinic Database**
##**Name:** Harmain Raza
##**Student Id:** 25023333

###**Install and imports**

In [ ]:
!pip install faker -q

In [ ]:
import sqlite3, random
from faker import Faker

# Fixed seeds make the "random" data reproducible, so anyone
# running this notebook gets the exact same database.
SEED = 42
random.seed(SEED)
fake = Faker("en_GB")
Faker.seed(SEED)

DB_PATH = "clinic.db"

###**Create the schema**

In [ ]:
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

# SQLite does NOT enforce foreign keys unless this is switched on.
cur.execute("PRAGMA foreign_keys = ON;")

# Drop tables in child-to-parent order so the notebook can be re-run cleanly.
for t in ["prescriptions", "appointments", "medications",
          "patients", "doctors", "departments"]:
    cur.execute(f"DROP TABLE IF EXISTS {t};")

cur.executescript("""
-- Departments are separated out so department details are stored once,
-- not repeated on every doctor row (avoids redundancy / update anomalies).
CREATE TABLE departments (
    department_id    INTEGER PRIMARY KEY,
    department_name  TEXT NOT NULL UNIQUE,            -- nominal
    floor_number     INTEGER CHECK (floor_number BETWEEN 0 AND 10)
);

CREATE TABLE doctors (
    doctor_id        INTEGER PRIMARY KEY,
    full_name        TEXT NOT NULL,
    department_id    INTEGER NOT NULL,
    years_experience INTEGER CHECK (years_experience BETWEEN 0 AND 45),  -- ratio
    seniority        TEXT NOT NULL
        CHECK (seniority IN ('Junior','Registrar','Consultant')),        -- ordinal
    FOREIGN KEY (department_id) REFERENCES departments(department_id)
);

CREATE TABLE patients (
    patient_id        INTEGER PRIMARY KEY,   -- pseudonymous ID, no national ID stored
    full_name         TEXT NOT NULL,         -- synthetic (Faker)
    gender            TEXT NOT NULL CHECK (gender IN ('Female','Male','Other')),   -- nominal
    birth_date        TEXT NOT NULL,                                               -- interval (calendar date)
    blood_type        TEXT NOT NULL
        CHECK (blood_type IN ('O+','O-','A+','A-','B+','B-','AB+','AB-')),         -- nominal
    city              TEXT NOT NULL,         -- city only (data minimisation)
    height_cm         REAL CHECK (height_cm BETWEEN 40 AND 230),                   -- ratio
    weight_kg         REAL CHECK (weight_kg BETWEEN 2 AND 250),                    -- ratio
    smoking_status    TEXT CHECK (smoking_status IN ('Never','Former','Current')), -- ordinal
    registration_date TEXT NOT NULL
);

CREATE TABLE medications (
    medication_id INTEGER PRIMARY KEY,
    drug_name     TEXT NOT NULL UNIQUE,
    drug_class    TEXT NOT NULL,             -- nominal
    unit_cost     REAL NOT NULL CHECK (unit_cost > 0)   -- ratio
);

CREATE TABLE appointments (
    appointment_id       INTEGER PRIMARY KEY,
    patient_id           INTEGER NOT NULL,
    doctor_id            INTEGER NOT NULL,
    appointment_datetime TEXT NOT NULL,                                  -- interval
    triage_level         TEXT NOT NULL
        CHECK (triage_level IN ('Low','Medium','High','Critical')),      -- ordinal
    body_temperature_c   REAL CHECK (body_temperature_c BETWEEN 34 AND 42),  -- interval (0 C is arbitrary)
    wait_time_minutes    INTEGER CHECK (wait_time_minutes >= 0),         -- ratio
    consultation_fee     REAL NOT NULL CHECK (consultation_fee >= 0),    -- ratio
    payment_method       TEXT NOT NULL
        CHECK (payment_method IN ('Insurance','Card','Cash','Free')),    -- nominal
    satisfaction_rating  INTEGER CHECK (satisfaction_rating BETWEEN 1 AND 5),  -- ordinal, NULL allowed
    FOREIGN KEY (patient_id) REFERENCES patients(patient_id),
    FOREIGN KEY (doctor_id)  REFERENCES doctors(doctor_id)
);

-- Junction table resolving the many-to-many link between appointments and
-- medications. The COMPOUND primary key stops the same drug being
-- prescribed twice in one appointment.
CREATE TABLE prescriptions (
    appointment_id INTEGER NOT NULL,
    medication_id  INTEGER NOT NULL,
    dosage_mg      REAL NOT NULL CHECK (dosage_mg > 0),
    duration_days  INTEGER NOT NULL CHECK (duration_days BETWEEN 1 AND 365),
    PRIMARY KEY (appointment_id, medication_id),
    FOREIGN KEY (appointment_id) REFERENCES appointments(appointment_id),
    FOREIGN KEY (medication_id)  REFERENCES medications(medication_id)
);

-- Indexes on foreign keys speed up the joins we will run later.
CREATE INDEX idx_doctors_dept   ON doctors(department_id);
CREATE INDEX idx_appt_patient   ON appointments(patient_id);
CREATE INDEX idx_appt_doctor    ON appointments(doctor_id);
CREATE INDEX idx_rx_medication  ON prescriptions(medication_id);
""")
conn.commit()
print("Schema created.")

Schema created.


###**Fill the lookup tables (departments, doctors, medications)**

In [ ]:
import re

# Matches a leading title such as Mr, Mrs, Miss, Ms, Dr, Prof (optional dot).
TITLE_RE = re.compile(r"^((Mr|Mrs|Miss|Ms|Mx|Dr|Prof|Rev|Sir|Lady|Lord)\.?\s+)+", re.I)

pat_bad = cur.execute(
    "SELECT COUNT(*) FROM patients WHERE full_name GLOB '[MDPRSL]*' "
    "AND (full_name LIKE 'Mr %' OR full_name LIKE 'Mrs %' OR full_name LIKE 'Miss %' "
    "OR full_name LIKE 'Ms %' OR full_name LIKE 'Dr %' OR full_name LIKE 'Prof %')"
).fetchone()[0]
doc_bad = cur.execute("SELECT COUNT(*) FROM doctors WHERE full_name LIKE 'Dr. %.%' "
                      "OR full_name LIKE 'Dr. Dr%' OR full_name LIKE 'Dr. Mr%' "
                      "OR full_name LIKE 'Dr. Mrs%' OR full_name LIKE 'Dr. Miss%' "
                      "OR full_name LIKE 'Dr. Ms%' OR full_name LIKE 'Dr. Prof%'").fetchone()[0]
print("Patients with titles:", pat_bad)
print("Doctors with doubled titles:", doc_bad)

Patients with titles: 0
Doctors with doubled titles: 0


In [ ]:
def clean_name(name):
    """Strip any leading honorific so names are plain 'First Last'."""
    return TITLE_RE.sub("", name).strip()

# Patients: titles removed entirely (a patient register normally stores plain names).
for pid, name in cur.execute("SELECT patient_id, full_name FROM patients").fetchall():
    cleaned = clean_name(name)
    if cleaned != name:
        cur.execute("UPDATE patients SET full_name=? WHERE patient_id=?", (cleaned, pid))

# Doctors: remove all titles, then add a single 'Dr. ' prefix.
for did, name in cur.execute("SELECT doctor_id, full_name FROM doctors").fetchall():
    cleaned = "Dr. " + clean_name(name.replace("Dr. ", "", 1))
    if cleaned != name:
        cur.execute("UPDATE doctors SET full_name=? WHERE doctor_id=?", (cleaned, did))

conn.commit()

print("Patient samples:", cur.execute("SELECT full_name FROM patients LIMIT 5").fetchall())
print("Doctor samples: ", cur.execute("SELECT full_name FROM doctors LIMIT 5").fetchall())
print("Patients still with titles:",
      cur.execute("SELECT COUNT(*) FROM patients WHERE full_name LIKE 'Mr %' OR full_name LIKE 'Mrs %' "
                  "OR full_name LIKE 'Miss %' OR full_name LIKE 'Ms %' OR full_name LIKE 'Dr %' "
                  "OR full_name LIKE 'Prof %'").fetchone()[0])

Patient samples: []
Doctor samples:  []
Patients still with titles: 0


In [ ]:
# Departments (hand-written: a small, fixed reference list)
departments = [
    ("Cardiology", 3), ("Pediatrics", 1), ("Orthopedics", 2), ("Neurology", 4),
    ("Dermatology", 2), ("General Medicine", 0), ("Oncology", 5), ("Psychiatry", 4),
]
cur.executemany(
    "INSERT INTO departments (department_name, floor_number) VALUES (?, ?)",
    departments)

# --- Doctors ---
# Weight departments so busy ones (General Medicine) get more doctors.
dept_ids     = list(range(1, len(departments) + 1))
dept_weights = [10, 8, 8, 6, 6, 18, 5, 6]

doctor_rows = []
for _ in range(60):
    # Seniority is chosen first, and experience is drawn to fit it,
    # so the ordinal and ratio columns stay consistent with each other.
    seniority = random.choices(["Junior", "Registrar", "Consultant"],
                               weights=[35, 35, 30])[0]
    exp_range = {"Junior": (0, 5), "Registrar": (5, 12), "Consultant": (12, 40)}
    years = random.randint(*exp_range[seniority])
    doctor_rows.append((
        "Dr. " + clean_name(fake.name()),
        random.choices(dept_ids, weights=dept_weights)[0],
        years, seniority))
cur.executemany(
    "INSERT INTO doctors (full_name, department_id, years_experience, seniority) "
    "VALUES (?, ?, ?, ?)", doctor_rows)

# --- Medications (hand-written, realistic names, classes and costs in GBP) ---
medications = [
    ("Amoxicillin","Antibiotic",4.50), ("Azithromycin","Antibiotic",6.80),
    ("Doxycycline","Antibiotic",5.20), ("Ciprofloxacin","Antibiotic",7.10),
    ("Ibuprofen","Analgesic",2.10), ("Paracetamol","Analgesic",1.20),
    ("Naproxen","Analgesic",3.40), ("Tramadol","Analgesic",8.90),
    ("Atorvastatin","Cardiovascular",5.60), ("Amlodipine","Cardiovascular",3.90),
    ("Ramipril","Cardiovascular",4.20), ("Bisoprolol","Cardiovascular",4.00),
    ("Warfarin","Cardiovascular",6.30), ("Clopidogrel","Cardiovascular",7.50),
    ("Metformin","Antidiabetic",3.10), ("Gliclazide","Antidiabetic",4.40),
    ("Sitagliptin","Antidiabetic",18.00), ("Insulin glargine","Antidiabetic",32.00),
    ("Salbutamol","Respiratory",4.80), ("Beclometasone","Respiratory",9.50),
    ("Montelukast","Respiratory",8.20), ("Cetirizine","Antihistamine",2.30),
    ("Loratadine","Antihistamine",2.60), ("Omeprazole","Gastrointestinal",3.70),
    ("Lansoprazole","Gastrointestinal",4.10), ("Ondansetron","Gastrointestinal",6.40),
    ("Sertraline","Antidepressant",5.90), ("Fluoxetine","Antidepressant",5.30),
    ("Citalopram","Antidepressant",5.10), ("Diazepam","Anxiolytic",4.70),
    ("Levothyroxine","Endocrine",3.80), ("Prednisolone","Corticosteroid",4.30),
    ("Hydrocortisone cream","Dermatological",3.50), ("Clobetasol","Dermatological",6.20),
    ("Isotretinoin","Dermatological",24.00), ("Gabapentin","Neurological",7.80),
    ("Carbamazepine","Neurological",6.60), ("Sumatriptan","Neurological",9.10),
    ("Tamoxifen","Oncology",14.50), ("Anastrozole","Oncology",16.80),
]
cur.executemany(
    "INSERT INTO medications (drug_name, drug_class, unit_cost) VALUES (?, ?, ?)",
    medications)

conn.commit()

In [ ]:
for t in ["departments", "doctors", "medications"]:
    n = cur.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"{t}: {n} rows")

print("\nDoctors per department:")
for row in cur.execute("""
    SELECT d.department_name, COUNT(*)
    FROM doctors doc JOIN departments d USING (department_id)
    GROUP BY d.department_name ORDER BY 2 DESC"""):
    print(row)

print("\nFK violations:", cur.execute("PRAGMA foreign_key_check").fetchall())

departments: 8 rows
doctors: 60 rows
medications: 40 rows

Doctors per department:
('General Medicine', 16)
('Pediatrics', 10)
('Cardiology', 8)
('Psychiatry', 8)
('Dermatology', 7)
('Oncology', 4)
('Orthopedics', 4)
('Neurology', 3)

FK violations: []


###**Generate and insert patients**

In [ ]:
from datetime import date, timedelta

In [ ]:

REF_DATE = date(2025, 6, 30)
N_PATIENTS = 1500

# Real-world-style blood type frequencies (O+ and A+ common, AB- rare).
blood_types   = ["O+", "A+", "B+", "AB+", "O-", "A-", "B-", "AB-"]
blood_weights = [35,   30,   8,    2,     13,   8,    2,    1]

# Weighted list of cities: bigger cities get more patients.
cities = ["London", "Birmingham", "Manchester", "Leeds", "Liverpool",
          "Bristol", "Sheffield", "Newcastle", "Nottingham", "Leicester"]
city_weights = [30, 14, 12, 8, 7, 7, 6, 6, 5, 5]

def make_patient():
    # --- Gender (nominal): small 'Other' share ---
    gender = random.choices(["Female", "Male", "Other"], weights=[50, 49, 1])[0]

    # Gender-matched fake names so names look realistic.
    if gender == "Female":
        name = clean_name(fake.name_female())
    elif gender == "Male":
        name = fake.name_male()
    else:
        name = fake.name()

    # --- Age: ~18% children, otherwise adults centred on mid-40s ---
    if random.random() < 0.18:
        age = random.uniform(0, 15.99)
    else:
        age = min(max(random.gauss(48, 18), 16), 95)   # clip to plausible range
    birth = REF_DATE - timedelta(days=int(age * 365.25))

    # --- Height (ratio): children by age, adults by gender (normal dist.) ---
    if age < 2:
        height = 50 + 20 * age
    elif age < 16:
        height = 90 + 5.2 * (age - 2)
    else:
        mean, sd = {"Male": (175.5, 7), "Female": (162, 6.5), "Other": (168, 8)}[gender]
        height = random.gauss(mean, sd)
    if age < 16:
        height += random.gauss(0, 4)                   # natural variation

    # --- Weight (ratio): derived from a random BMI so weight tracks height ---
    bmi_mean, bmi_sd = (16, 2) if age < 16 else (26.5, 4.5)
    bmi = max(random.gauss(bmi_mean, bmi_sd), 12)
    weight = bmi * (height / 100) ** 2

    # Clamp to the CHECK-constraint ranges so rare outliers never break inserts.
    height = round(min(max(height, 45), 220), 1)
    weight = round(min(max(weight, 2.5), 200), 1)

    # --- Smoking status (ordinal): only for 16+, ~3% missing (NULL) ---
    if age < 16:
        smoking = "Never"
    elif random.random() < 0.03:
        smoking = None
    else:
        smoking = random.choices(["Never", "Former", "Current"],
                                 weights=[60, 22, 18])[0]

    # --- Registration date: random day in 2023-01-01..2025-06-30,
    #     but never before the patient was born. ---
    reg = date(2023, 1, 1) + timedelta(days=random.randint(0, 910))
    reg = max(reg, birth)

    return (name, gender, birth.isoformat(),
            random.choices(blood_types, weights=blood_weights)[0],
            random.choices(cities, weights=city_weights)[0],
            height, weight, smoking, reg.isoformat())

patient_rows = [make_patient() for _ in range(N_PATIENTS)]
cur.executemany("""
    INSERT INTO patients (full_name, gender, birth_date, blood_type, city,
                          height_cm, weight_kg, smoking_status, registration_date)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)""", patient_rows)
conn.commit()
print("Inserted", N_PATIENTS, "patients")

Inserted 1500 patients


###**Validate the patients table**

In [ ]:
print("Row count:", cur.execute("SELECT COUNT(*) FROM patients").fetchone()[0])

print("\nGender split:")
for r in cur.execute("SELECT gender, COUNT(*) FROM patients GROUP BY gender"): print(r)

print("\nBlood types (should be O+ and A+ most common):")
for r in cur.execute("SELECT blood_type, COUNT(*) FROM patients GROUP BY blood_type ORDER BY 2 DESC"): print(r)

print("\nSmoking status:")
for r in cur.execute("SELECT smoking_status, COUNT(*) FROM patients GROUP BY smoking_status"): print(r)

print("\nAverage height/weight by gender (adults 16+):")
for r in cur.execute("""
    SELECT gender, ROUND(AVG(height_cm),1), ROUND(AVG(weight_kg),1)
    FROM patients
    WHERE birth_date <= date('2009-06-30')
    GROUP BY gender"""): print(r)

print("\nChildren (under 16):",
      cur.execute("SELECT COUNT(*) FROM patients WHERE birth_date > date('2009-06-30')").fetchone()[0])

print("\nRegistered before birth (should be 0):",
      cur.execute("SELECT COUNT(*) FROM patients WHERE registration_date < birth_date").fetchone()[0])

print("\nSample rows:")
for r in cur.execute("SELECT * FROM patients LIMIT 5"): print(r)

Row count: 1500

Gender split:
('Female', 741)
('Male', 738)
('Other', 21)

Blood types (should be O+ and A+ most common):
('O+', 530)
('A+', 459)
('O-', 185)
('A-', 138)
('B+', 114)
('B-', 32)
('AB+', 28)
('AB-', 14)

Smoking status:
(None, 34)
('Current', 206)
('Former', 249)
('Never', 1011)

Average height/weight by gender (adults 16+):
('Female', 161.7, 69.4)
('Male', 175.1, 81.3)
('Other', 170.2, 76.6)

Children (under 16): 275

Registered before birth (should be 0): 0

Sample rows:
(1, 'Paul Hammond', 'Male', '1965-07-09', 'A+', 'London', 177.7, 78.9, 'Current', '2024-08-07')
(2, 'Pauline Townsend', 'Female', '1988-05-30', 'B+', 'Liverpool', 169.9, 87.1, 'Never', '2025-06-03')
(3, 'Roger Bruce', 'Male', '2019-01-22', 'O+', 'Manchester', 108.8, 22.9, 'Never', '2023-09-12')
(4, 'Olivia King', 'Female', '2009-06-30', 'AB-', 'Sheffield', 165.1, 92.0, 'Never', '2024-10-29')
(5, 'Mathew Daniels', 'Male', '1962-07-31', 'A-', 'London', 178.0, 76.9, 'Never', '2024-02-15')


###**Generate and insert appointments**

In [ ]:
from datetime import datetime

N_APPOINTMENTS = 5000
END_DATE = date(2025, 12, 31)   # appointments run from each patient's registration to here

# Load parent data once so the generator never queries inside the loop.
patients = [(pid, date.fromisoformat(b), date.fromisoformat(r))
            for pid, b, r in cur.execute(
                "SELECT patient_id, birth_date, registration_date FROM patients")]

docs_by_dept = {}
for did, dept, sen in cur.execute("SELECT doctor_id, department_id, seniority FROM doctors"):
    docs_by_dept.setdefault(dept, []).append((did, sen))

# Department ids follow insertion order in Cell 3:
# 1 Cardiology, 2 Pediatrics, 3 Orthopedics, 4 Neurology,
# 5 Dermatology, 6 General Medicine, 7 Oncology, 8 Psychiatry

# Triage weights [Low, Medium, High, Critical] per department:
# clinical severity differs by speciality, so they are not all equal.
TRIAGE = ["Low", "Medium", "High", "Critical"]
triage_weights = {
    1: [20, 35, 30, 15], 2: [55, 32, 10, 3], 3: [40, 38, 17, 5], 4: [25, 40, 25, 10],
    5: [70, 25, 4, 1],   6: [50, 35, 12, 3], 7: [15, 35, 32, 18], 8: [30, 45, 20, 5]}

# Mean wait (minutes) by triage: urgent patients are seen faster, as in real triage.
mean_wait = {"Low": 45, "Medium": 30, "High": 15, "Critical": 5}

base_fee = {1: 120, 2: 60, 3: 100, 4: 130, 5: 80, 6: 50, 7: 150, 8: 90}   # GBP
seniority_factor = {"Junior": 0.8, "Registrar": 1.0, "Consultant": 1.4}
triage_factor    = {"Low": 1.0, "Medium": 1.1, "High": 1.25, "Critical": 1.5}

# Some patients are "frequent attenders" (chronic conditions), so patients are
# sampled with unequal weights instead of uniformly.
propensity = [random.expovariate(1.0) + 0.1 for _ in patients]
chosen = random.choices(patients, weights=propensity, k=N_APPOINTMENTS)

def make_appointment(pat):
    pid, birth, reg = pat

    # --- Date: uniform between registration and END_DATE, weekdays only ---
    d = reg + timedelta(days=random.randint(0, (END_DATE - reg).days))
    if d.weekday() >= 5:                       # Sat/Sun -> following Monday
        d += timedelta(days=7 - d.weekday())
    hour = random.choices(range(8, 18), weights=[8, 12, 12, 10, 6, 8, 10, 10, 8, 4])[0]
    minute = random.choice([0, 15, 30, 45])
    dt = datetime(d.year, d.month, d.day, hour, minute)

    age = (d - birth).days / 365.25

    # Department depends on age
    if age < 16:
        dept = random.choices([2, 6, 5], weights=[90, 7, 3])[0]
    else:
        w = [12, 10, 7, 8, 30, 6, 8]           # Cardio, Ortho, Neuro, Derm, GM, Onc, Psych
        if age >= 55:
            w[0], w[5] = 22, 10                # older adults: more Cardiology/Oncology
        dept = random.choices([1, 3, 4, 5, 6, 7, 8], weights=w)[0]

    doctor_id, seniority = random.choice(docs_by_dept[dept])

    # Triage (ordinal)
    triage = random.choices(TRIAGE, weights=triage_weights[dept])[0]

    # Body temperature (interval): normal ~36.8, occasional fever
    fever_p = 0.12 if dept == 2 else 0.06
    if triage == "Critical":
        fever_p += 0.25                        # critical cases run hotter more often
    if random.random() < fever_p:
        temp = random.gauss(38.4, 0.7)
    else:
        temp = random.gauss(36.8, 0.3)
    temp = round(min(max(temp, 34.5), 41.5), 1)
    if random.random() < 0.02:
        temp = None                            # not recorded

    # Wait time (ratio): exponential, capped at 4 hours
    wait = min(int(random.expovariate(1 / mean_wait[triage])), 240)

    # Payment (nominal): children are more often covered for free
    if age < 16:
        pay = random.choices(["Free", "Insurance", "Card", "Cash"], weights=[50, 35, 10, 5])[0]
    else:
        pay = random.choices(["Insurance", "Card", "Cash", "Free"], weights=[45, 25, 15, 15])[0]

    # Fee (ratio): department x seniority x triage, with small noise; Free = 0
    if pay == "Free":
        fee = 0.0
    else:
        fee = round(base_fee[dept] * seniority_factor[seniority]
                    * triage_factor[triage] * random.gauss(1, 0.05), 2)

    # Satisfaction (ordinal 1-5): drops as waiting rises; ~25% never answered
    if random.random() < 0.25:
        sat = None
    else:
        sat = int(min(max(round(random.gauss(4.3 - wait / 50, 0.8)), 1), 5))

    return (pid, doctor_id, dt.strftime("%Y-%m-%d %H:%M"),
            triage, temp, wait, fee, pay, sat)

rows = [make_appointment(p) for p in chosen]
rows.sort(key=lambda r: r[2])                  # chronological, so IDs increase with time

cur.executemany("""
    INSERT INTO appointments (patient_id, doctor_id, appointment_datetime, triage_level,
        body_temperature_c, wait_time_minutes, consultation_fee, payment_method,
        satisfaction_rating)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)""", rows)
conn.commit()
print("Inserted", len(rows), "appointments")

Inserted 5000 appointments


In [ ]:
q = lambda sql: cur.execute(sql).fetchall()

print("Rows:", q("SELECT COUNT(*) FROM appointments")[0][0])
print("Date range:", q("SELECT MIN(appointment_datetime), MAX(appointment_datetime) FROM appointments"))

print("\nTriage distribution:", q("SELECT triage_level, COUNT(*) FROM appointments GROUP BY 1"))

print("\nAvg wait by triage (expect Low > Medium > High > Critical):")
for r in q("""SELECT triage_level, ROUND(AVG(wait_time_minutes),1) FROM appointments
              GROUP BY 1 ORDER BY 2 DESC"""): print(r)

print("\nAvg fee by department (paid visits only):")
for r in q("""SELECT d.department_name, ROUND(AVG(a.consultation_fee),2)
              FROM appointments a JOIN doctors doc USING (doctor_id)
              JOIN departments d USING (department_id)
              WHERE a.payment_method <> 'Free' GROUP BY 1 ORDER BY 2 DESC"""): print(r)

print("\nAvg satisfaction by wait band (expect it to fall):")
for r in q("""SELECT CASE WHEN wait_time_minutes < 15 THEN '1) <15'
                         WHEN wait_time_minutes < 45 THEN '2) 15-44' ELSE '3) 45+' END,
                     ROUND(AVG(satisfaction_rating),2)
              FROM appointments GROUP BY 1 ORDER BY 1"""): print(r)

print("\nNULL satisfaction %:", q("SELECT ROUND(100.0*SUM(satisfaction_rating IS NULL)/COUNT(*),1) FROM appointments")[0][0])
print("NULL temperature %: ", q("SELECT ROUND(100.0*SUM(body_temperature_c IS NULL)/COUNT(*),1) FROM appointments")[0][0])
print("Fever (>=38.0) %:   ", q("SELECT ROUND(100.0*SUM(body_temperature_c>=38.0)/COUNT(*),1) FROM appointments")[0][0])

# Integrity checks (all should be 0)
print("\nBooked before registration:", q("""SELECT COUNT(*) FROM appointments a JOIN patients p USING (patient_id)
      WHERE date(a.appointment_datetime) < p.registration_date""")[0][0])
print("Adults (16+) in Pediatrics:", q("""SELECT COUNT(*) FROM appointments a JOIN patients p USING (patient_id)
      JOIN doctors d USING (doctor_id) JOIN departments dep USING (department_id)
      WHERE dep.department_name='Pediatrics'
      AND date(a.appointment_datetime) >= date(p.birth_date,'+16 years')""")[0][0])
print("Weekend appointments:", q("""SELECT COUNT(*) FROM appointments
      WHERE strftime('%w', appointment_datetime) IN ('0','6')""")[0][0])
print("FK violations:", q("PRAGMA foreign_key_check"))

print("\nBusiest patients:", q("""SELECT patient_id, COUNT(*) c FROM appointments GROUP BY 1 ORDER BY c DESC LIMIT 3"""))

Rows: 5000
Date range: [('2023-01-20 08:45', '2025-12-31 16:30')]

Triage distribution: [('Critical', 328), ('High', 874), ('Low', 2046), ('Medium', 1752)]

Avg wait by triage (expect Low > Medium > High > Critical):
('Low', 45.9)
('Medium', 29.7)
('High', 14.1)
('Critical', 4.8)

Avg fee by department (paid visits only):
('Oncology', 179.9)
('Cardiology', 148.74)
('Orthopedics', 127.81)
('Neurology', 119.06)
('Psychiatry', 112.63)
('Dermatology', 94.07)
('Pediatrics', 67.34)
('General Medicine', 57.88)

Avg satisfaction by wait band (expect it to fall):
('1) <15', 4.15)
('2) 15-44', 3.75)
('3) 45+', 2.58)

NULL satisfaction %: 23.8
NULL temperature %:  2.0
Fever (>=38.0) %:    6.4

Booked before registration: 0
Adults (16+) in Pediatrics: 0
Weekend appointments: 0
FK violations: []

Busiest patients: [(454, 28), (77, 21), (210, 21)]


###**Generate and insert prescriptions**

In [ ]:
# Group medication ids by class so each department can draw from suitable drugs.
meds_by_class = {}
for mid, cls in cur.execute("SELECT medication_id, drug_class FROM medications"):
    meds_by_class.setdefault(cls, []).append(mid)
med_class = {m: c for c, ids in meds_by_class.items() for m in ids}

# Which drug classes each department prescribes, with relative weights.
# Real clinics prescribe according to speciality, not at random.
dept_classes = {
    "Cardiology":       {"Cardiovascular": 80, "Analgesic": 10, "Antidiabetic": 10},
    "Pediatrics":       {"Antibiotic": 35, "Antihistamine": 20, "Analgesic": 25, "Respiratory": 20},
    "Orthopedics":      {"Analgesic": 70, "Corticosteroid": 30},
    "Neurology":        {"Neurological": 75, "Analgesic": 25},
    "Dermatology":      {"Dermatological": 60, "Antibiotic": 20, "Antihistamine": 20},
    "General Medicine": {"Antibiotic": 20, "Analgesic": 20, "Antidiabetic": 12, "Cardiovascular": 12,
                         "Respiratory": 10, "Gastrointestinal": 12, "Endocrine": 7, "Antihistamine": 7},
    "Oncology":         {"Oncology": 60, "Analgesic": 25, "Gastrointestinal": 15},
    "Psychiatry":       {"Antidepressant": 65, "Anxiolytic": 35},
}

# Typical adult doses (mg) per class (simplified, but plausible).
dose_options = {
    "Antibiotic": [250, 500, 875], "Analgesic": [200, 400, 500, 600],
    "Cardiovascular": [5, 10, 20, 40, 75], "Antidiabetic": [500, 850, 1000],
    "Respiratory": [10, 100, 200], "Antihistamine": [5, 10],
    "Gastrointestinal": [4, 20, 30], "Antidepressant": [20, 50, 100],
    "Anxiolytic": [2, 5, 10], "Endocrine": [0.05, 0.1, 0.15],
    "Corticosteroid": [5, 20, 40], "Dermatological": [10, 25, 50],
    "Neurological": [100, 300, 600], "Oncology": [1, 20, 25],
}
# Course length (days) per class: acute drugs are short, chronic ones long.
duration_range = {
    "Antibiotic": (5, 14), "Analgesic": (3, 14), "Cardiovascular": (30, 180),
    "Antidiabetic": (30, 180), "Respiratory": (14, 90), "Antihistamine": (7, 30),
    "Gastrointestinal": (14, 56), "Antidepressant": (30, 180), "Anxiolytic": (7, 28),
    "Endocrine": (90, 365), "Corticosteroid": (5, 21), "Dermatological": (7, 42),
    "Neurological": (30, 180), "Oncology": (90, 365),
}
# Chance an appointment results in a prescription, by triage level.
rx_prob = {"Low": 0.45, "Medium": 0.65, "High": 0.80, "Critical": 0.85}

appts = cur.execute("""
    SELECT a.appointment_id, a.triage_level, dep.department_name,
           p.birth_date, p.weight_kg, a.appointment_datetime
    FROM appointments a
    JOIN patients p USING (patient_id)
    JOIN doctors d USING (doctor_id)
    JOIN departments dep USING (department_id)""").fetchall()

rx_rows = []
for appt_id, triage, dept, birth, weight, when in appts:
    if random.random() > rx_prob[triage]:
        continue                                    # no prescription this visit

    age = (date.fromisoformat(when[:10]) - date.fromisoformat(birth)).days / 365.25
    n_drugs = random.choices([1, 2, 3], weights=[45, 35, 20])[0]

    chosen_meds = set()                             # a set => no repeats within a visit
    classes, weights = zip(*dept_classes[dept].items())
    while len(chosen_meds) < n_drugs:
        cls = random.choices(classes, weights=weights)[0]
        chosen_meds.add(random.choice(meds_by_class[cls]))

    for mid in chosen_meds:
        cls = med_class[mid]
        dose = random.choice(dose_options[cls])
        if age < 16:                                # paediatric dose scales with weight
            dose *= max(weight / 70, 0.1)
        dose = round(dose, 2)

        days = random.randint(*duration_range[cls])
        if days >= 28:
            days = min(round(days / 7) * 7, 365)    # long courses come in whole weeks
        rx_rows.append((appt_id, mid, dose, days))

cur.executemany("""INSERT INTO prescriptions
    (appointment_id, medication_id, dosage_mg, duration_days) VALUES (?, ?, ?, ?)""", rx_rows)
conn.commit()
print("Inserted", len(rx_rows), "prescriptions")

Inserted 5345 prescriptions


###**Full-database validation**

In [ ]:
cur.execute("PRAGMA foreign_keys = ON;")   # re-assert in case the connection was restarted
q = lambda sql: cur.execute(sql).fetchall()

print("=== Row counts ===")
total = 0
for t in ["departments", "doctors", "patients", "medications", "appointments", "prescriptions"]:
    n = q(f"SELECT COUNT(*) FROM {t}")[0][0]; total += n
    print(f"{t:15s}{n:>6}")
print(f"{'TOTAL':15s}{total:>6}")

print("\nAppointments with a prescription (%):",
      q("SELECT ROUND(100.0*COUNT(DISTINCT appointment_id)/(SELECT COUNT(*) FROM appointments),1) FROM prescriptions")[0][0])

print("\nTop drug classes by department (sanity check):")
for r in q("""SELECT dep.department_name, m.drug_class, COUNT(*) c
              FROM prescriptions rx JOIN medications m USING (medication_id)
              JOIN appointments a USING (appointment_id)
              JOIN doctors d USING (doctor_id) JOIN departments dep USING (department_id)
              WHERE dep.department_name IN ('Psychiatry','Dermatology','Oncology')
              GROUP BY 1,2 ORDER BY 1, c DESC"""): print(r)

print("\nAvg dose, children vs adults for Amoxicillin:")
for r in q("""SELECT CASE WHEN a.appointment_datetime < date(p.birth_date,'+16 years')
                       THEN 'child' ELSE 'adult' END, ROUND(AVG(rx.dosage_mg),1), COUNT(*)
              FROM prescriptions rx JOIN medications m USING (medication_id)
              JOIN appointments a USING (appointment_id) JOIN patients p USING (patient_id)
              WHERE m.drug_name='Amoxicillin' GROUP BY 1"""): print(r)

print("\nMost prescribed drugs:", q("""SELECT m.drug_name, COUNT(*) FROM prescriptions rx
      JOIN medications m USING (medication_id) GROUP BY 1 ORDER BY 2 DESC LIMIT 5"""))

# --- Prove the constraints actually work (useful evidence for the report) ---
import sqlite3
a, m = q("SELECT appointment_id, medication_id FROM prescriptions LIMIT 1")[0]
for label, params in [("Duplicate compound key", (a, m, 10, 5)),
                      ("Non-existent appointment (FK)", (999999, m, 10, 5)),
                      ("Negative dosage (CHECK)", (a, 40 if m != 40 else 39, -5, 5))]:
    try:
        cur.execute("INSERT INTO prescriptions VALUES (?,?,?,?)", params)
        print(f"{label}: NOT blocked (problem!)")
    except sqlite3.IntegrityError as e:
        print(f"{label}: blocked -> {e}")
conn.rollback()

print("\nFK violations:", q("PRAGMA foreign_key_check"))
print("Integrity check:", q("PRAGMA integrity_check"))

=== Row counts ===
departments         8
doctors            60
patients         1500
medications        40
appointments     5000
prescriptions    5345
TOTAL           11953

Appointments with a prescription (%): 60.8

Top drug classes by department (sanity check):
('Dermatology', 'Dermatological', 207)
('Dermatology', 'Antibiotic', 82)
('Dermatology', 'Antihistamine', 78)
('Oncology', 'Oncology', 249)
('Oncology', 'Analgesic', 139)
('Oncology', 'Gastrointestinal', 67)
('Psychiatry', 'Antidepressant', 299)
('Psychiatry', 'Anxiolytic', 130)

Avg dose, children vs adults for Amoxicillin:
('adult', 519.4, 90)
('child', 281.0, 87)

Most prescribed drugs: [('Paracetamol', 350), ('Tramadol', 312), ('Ibuprofen', 277), ('Naproxen', 275), ('Ciprofloxacin', 190)]
Duplicate compound key: blocked -> UNIQUE constraint failed: prescriptions.appointment_id, prescriptions.medication_id
Non-existent appointment (FK): blocked -> FOREIGN KEY constraint failed
Negative dosage (CHECK): blocked -> CHECK cons

###**Final tidy-up and download**

In [ ]:
# Final sanity check
cur.execute("PRAGMA foreign_keys = ON;")
print("FK violations:", cur.execute("PRAGMA foreign_key_check").fetchall())
conn.commit()
cur.execute("VACUUM;")

# Print the schema as text .
for name, sql in cur.execute("SELECT name, sql FROM sqlite_master WHERE type='table'"):
    print(sql, "\n")

import os
print("File size (KB):", round(os.path.getsize(DB_PATH) / 1024, 1))

from google.colab import files
files.download(DB_PATH)

FK violations: []
CREATE TABLE departments (
    department_id    INTEGER PRIMARY KEY,
    department_name  TEXT NOT NULL UNIQUE,            -- nominal
    floor_number     INTEGER CHECK (floor_number BETWEEN 0 AND 10)
) 

CREATE TABLE doctors (
    doctor_id        INTEGER PRIMARY KEY,
    full_name        TEXT NOT NULL,
    department_id    INTEGER NOT NULL,
    years_experience INTEGER CHECK (years_experience BETWEEN 0 AND 45),  -- ratio
    seniority        TEXT NOT NULL
        CHECK (seniority IN ('Junior','Registrar','Consultant')),        -- ordinal
    FOREIGN KEY (department_id) REFERENCES departments(department_id)
) 

CREATE TABLE patients (
    patient_id        INTEGER PRIMARY KEY,   -- pseudonymous ID, no national ID stored
    full_name         TEXT NOT NULL,         -- synthetic (Faker)
    gender            TEXT NOT NULL CHECK (gender IN ('Female','Male','Other')),   -- nominal
    birth_date        TEXT NOT NULL,                                               -- inter

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>